In [4]:
import sys, os, importlib, warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sys.path.insert(0, os.path.abspath('../../model'))
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../.'))

import numpy as np
import cariaco_baseline_setups as cbs; importlib.reload(cbs)
from parscan_utils import run_single_point

# Does the broadcast d_e (Inflow) propagate to the foreign d_e (PhytoSinking)
# under update_vars? Vary d_e at fixed F_N and check, at t0:
#   supply flux   = F_N/d_e   (intra-component control — must work)
#   sinking coef  = w_sink/d_e (the cross-component broadcast->foreign path)
FN_test = 5.0
print(f"{'d_e':>6} | {'F_N/d_e exp':>11} {'supply t0':>10} | "
      f"{'w/d_e exp':>9} {'sink coef':>9} {'max dev':>9}")
for de in (50.0, 25.0):
    out = run_single_point(
        cbs.model_baseline, cbs.model_setup_baseline,
        scan_params={}, fixed_overrides={'Inflow__FN': FN_test, 'Inflow__de': de})
    supply0 = float(out['Inflow__input_value'].isel(time=0).values)
    sink0   = out['PhytoSinking__sinking_value'].isel(time=0).values
    P0      = out['Phytoplankton__biomass'].isel(time=0).values
    coef    = sink0 / P0                       # should equal w_sink/d_e, uniform
    print(f"{de:6.1f} | {FN_test/de:11.4f} {supply0:10.4f} | "
          f"{cbs.W_SINK/de:9.4f} {np.nanmean(coef):9.4f} "
          f"{np.nanmax(np.abs(coef - cbs.W_SINK/de)):9.1e}")

   d_e | F_N/d_e exp  supply t0 | w/d_e exp sink coef   max dev
  50.0 |      0.1000     0.1000 |    0.1000    0.1298   8.6e-02
  25.0 |      0.2000     0.2000 |    0.2000    0.2469   1.5e-01


In [7]:
out['PhytoSinking__sinking_value'].isel(time=0)

<xarray.DataArray 'PhytoSinking__sinking_value' (phyto: 12)> Size: 96B
array([0.00035146, 0.00032013, 0.00029376, 0.00027161, 0.0002531 ,
       0.00023775, 0.00022517, 0.00021498, 0.00020685, 0.00020045,
       0.00019549, 0.0001917 ])
Coordinates:
  * phyto    (phyto) float64 96B 0.5 0.862 1.486 2.562 ... 67.29 116.0 200.0
    time     float64 8B 0.0
Attributes:
    description:    output of flux value / 
    xso_store_out:  True

In [3]:
import xso, numpy as np, importlib
import cariaco_baseline_setups as cbs; importlib.reload(cbs)
from parscan_utils import run_single_point

deriv_setup = xso.update_setup(model=cbs.model_baseline,
                               old_setup=cbs.model_setup_baseline,
                               new_solver='deriv', new_time=[0.0, 1.0])
for de in (50.0, 25.0):
    out = run_single_point(cbs.model_baseline, deriv_setup, scan_params={},
                           fixed_overrides={'Inflow__FN': 5.0, 'Inflow__de': de})
    coef = out['PhytoSinking__sinking_value'].isel(time=0).values / \
           out['Phytoplankton__biomass'].isel(time=0).values
    print(f"de={de:5.1f}  w/de={cbs.W_SINK/de:.4f}  coef={np.nanmean(coef):.4f}  "
          f"max dev={np.nanmax(np.abs(coef - cbs.W_SINK/de)):.1e}")

de= 50.0  w/de=0.1000  coef=0.0000  max dev=1.0e-01
de= 25.0  w/de=0.2000  coef=0.0000  max dev=2.0e-01


In [1]:
import xso, numpy as np

@xso.component
class StateX:
    value = xso.variable(description='a state variable')

@xso.component
class SourceX:                      # owns the broadcast parameter, has a flux
    target   = xso.variable(foreign=True, flux='grow', negative=False)
    p_shared = xso.parameter(broadcast=True, description='broadcast parameter')

    @xso.flux
    def grow(self, target, p_shared):
        return p_shared            # constant source term = p_shared

@xso.component
class ConsumerX:                    # foreign-references the broadcast parameter
    target = xso.variable(foreign=True, flux='shrink', negative=True)
    p_ref  = xso.parameter(foreign=True, description='foreign ref to broadcast param')

    @xso.flux
    def shrink(self, target, p_ref):
        return p_ref * target

model = xso.create({'State': StateX, 'Source': SourceX, 'Consumer': ConsumerX})

setup = xso.setup(
    solver='solve_ivp', model=model, time=np.arange(0, 5, 0.1),
    input_vars={
        'State':    {'value_label': 'X', 'value_init': 1.0},
        'Source':   {'target': 'X', 'p_shared': 0.5, 'p_shared_label': 'shared_p'},
        'Consumer': {'target': 'X', 'p_ref': 'shared_p'},
    },
)
with model:
    out = setup.xsimlab.run()
print('final X =', float(out['State__value'].values[-1]))   # ODE: dX/dt = 0.5 - 0.5X -> X stays ~1

final X = 1.0
